# Grover's Search

This notebook demonstrates Grover's algorithm for unstructured search.
It uses an oracle defined in an auxiliary Q# file (`Oracle.qs`) in the same folder.

In [ ]:
from qdk import qsharp

In [ ]:
%%qsharp

import Std.Diagnostics.*;
import Std.Math.*;

operation GroverSearch(nQubits : Int, oracle : Qubit[] => Unit) : Result[] {
    let iterations = Round(PI() / 4.0 * Sqrt(IntAsDouble(1 <<< nQubits)));
    use register = Qubit[nQubits];

    // Initialize uniform superposition.
    for q in register { H(q); }

    for _ in 0..iterations - 1 {
        // Oracle.
        oracle(register);
        // Diffusion.
        Diffusion(register);
    }

    DumpMachine();
    let results = MeasureEachZ(register);
    ResetAll(register);
    results
}

operation Diffusion(register : Qubit[]) : Unit {
    for q in register { H(q); X(q); }
    Controlled Z(register[0..Length(register) - 2], register[Length(register) - 1]);
    for q in register { X(q); H(q); }
}

In [ ]:
%%qsharp

// Run Grover's search looking for |11⟩ in a 2-qubit space.
// MarkTarget is defined in the auxiliary Oracle.qs file.
operation Main() : Result[] {
    GroverSearch(2, MarkTarget)
}

Main()